# Compare custom model to LDA baseline

This example shows how to:
1. Run a custom model
2. Compare it to saved LDA baseline results with the corrected t-test

Prerequisites:
- Run the LDA baseline first, from the repository root: `uv run python scripts/classify_dataset.py -m lda --suite ma --seed 42`

### Imports

In [1]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from fnirsatlas.classification import run_benchmark
from fnirsatlas.compare import compare_models, format_comparison_table
from fnirsatlas.feature_extraction import WindowedFeatureExtractor
from fnirsatlas.results import Results
from fnirsatlas.utils import get_datasets_from_args

### Define custom model

Create a 3-step pipeline:
1. **WindowedFeatureExtractor**: Extracts standard deviation from each channel
2. **StandardScaler**: Z-normalizes features across samples
3. **SVC**: Support Vector Classifier with RBF kernel

In [2]:
def get_custom_svm(**kwargs):
    """Custom SVM with RBF kernel model."""
    random_state = kwargs["random_state"]
    return Pipeline(
        [
            (
                "feature_extractor",
                WindowedFeatureExtractor(n_windows=1, features=["std"]),
            ),
            ("scaler", StandardScaler()),
            (
                "clf",
                SVC(kernel="rbf", random_state=random_state, C=1.0, gamma="scale"),
            ),
        ]
    )

### Run the benchmark with the custom model

Run cross-subject classification on the MA suite with hyperparameter tuning using nested cross-validation (5-fold outer CV, 4-fold inner CV by default).

In [ ]:
suite = "ma"
seed = 42
metric = "cross_acc"

print("Running custom SVM with RBF kernel model...")
custom_results = run_benchmark(
    get_clf=get_custom_svm,
    suite=suite,
    classification_setting="cross",
    seed=seed,
    n_jobs=1,
    param_grid={
        "clf__C": [1e-3, 1, 1e3],
    },
)
custom_results["model_name"] = "Custom SVM RBF"

### Load baseline results

Load the saved LDA baseline results from `paths.results_cache`.

In [ ]:
lda_results = Results.gather_results(
    models=["lda"],
    datasets=get_datasets_from_args(None, suite),
    metric=metric,
    seeds=[seed],
    classification_setting="cross",
)
if not lda_results:
    raise FileNotFoundError(
        "LDA baseline not found. Run first:\n"
        f"    uv run python scripts/classify_dataset.py -m lda --suite {suite} --seed {seed}"
    )

### Compare custom model to LDA baseline

Perform statistical comparison using corrected t-test and FDR correction.

The comparison uses:
- **Corrected t-test** (Nadeau & Bengio, 2003) for cross-validation dependency
- **FDR correction** (Benjamini-Hochberg) for multiple comparisons  
- **Significance** based on FDR-corrected p-values

In [ ]:
comparisons = compare_models([custom_results, *lda_results], metric=metric, alpha=0.05)
for comparison_key, task_results in comparisons.items():
    format_comparison_table(comparison_key, task_results, alpha=0.05, metric=metric)

### Results Interpretation

- "Diff" column shows the accuracy difference between the two models
- "P-Orig" shows uncorrected p-values from corrected paired t-tests
- "P-FDR" shows False Discovery Rate corrected p-values (Benjamini-Hochberg)
- "Sig" shows significance levels: *** (p < 0.001), ** (p < 0.01), * (p < 0.05)